### ***Attention Is All You Need***

This notebook is a **from-scratch PyTorch implementation of the Transformer**, the architecture introduced in the paper [*Attention Is All You Need* (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762).

here's the big picture before diving into the code:

- Before Transformers, sequence models (like RNNs/LSTMs) processed text **one token at a time, in order**. This made them slow to train and bad at remembering long-range relationships between words.
- The Transformer's big idea is **attention** — instead of processing tokens one by one, every token looks directly at every other token in the sequence and decides how much to "pay attention" to it. This is done in parallel, which makes training much faster and lets the model capture long-range dependencies easily.
- The model has two main halves:
  - **Encoder** — reads the input sentence (e.g. a sentence in English) and builds a rich numerical representation of it.
  - **Decoder** — uses that representation to generate the output sentence (e.g. the French translation), one token at a time during inference.
- Both the encoder and decoder are built by **stacking identical blocks** on top of each other (6 blocks each, in the original paper). Each block is made of small, reusable pieces: **embeddings, positional encoding, multi-head attention, feed-forward layers, residual connections, and layer normalization.**

This notebook builds each of these pieces as its own PyTorch `nn.Module` class, from the bottom up, and finally assembles them into the full `Transformer` model. Each section below explains, in plain language, what that piece of code does and why it exists in the paper — the code itself is left exactly as implemented.


#### 1. Imports

Just the basic building blocks we need from PyTorch:

- `torch` — the core PyTorch library, gives us tensors (multi-dimensional arrays) and math operations on them.
- `torch.nn` (imported as `nn`) — PyTorch's neural network module. It gives us ready-made layers like `nn.Linear`, `nn.Embedding`, `nn.Dropout`, and the base class `nn.Module` that every custom layer/model in this notebook inherits from.
- `math` — Python's standard math library, used here for things like `math.sqrt` and `math.log` inside the positional encoding and attention formulas.


In [33]:
# all the imports
import torch
import torch.nn as nn
import math

#### 2. Input Embeddings

**What this does:** Converts input tokens (integers representing words/sub-words, e.g. `"cat"` → `42`) into dense vectors of size `d_model` that the model can actually do math with.

**Why it's needed:** Neural networks can't work directly with words or token IDs — they need continuous numbers. An *embedding layer* is essentially a lookup table: every possible token in the vocabulary gets its own learnable vector of size `d_model`. During training, the model adjusts these vectors so that tokens with similar meaning/usage end up with similar vectors.

**Key pieces:**
- `__init__(self, d_model, vocab_size)`:
  - `d_model` — the size of every embedding vector (and the size used throughout the whole model, e.g. 512 in the original paper).
  - `vocab_size` — how many unique tokens exist in the vocabulary.
  - `nn.Embedding(vocab_size, d_model)` — creates the actual lookup table of shape `(vocab_size, d_model)`.
- `forward(self, x)`:
  - `x` has shape `(batch_size, seq_len)` — a batch of sequences of token IDs.
  - `self.embedding(x)` looks up each token ID and returns its vector, giving shape `(batch_size, seq_len, d_model)`.
  - The result is multiplied by `sqrt(d_model)`. This is a detail straight from the paper: it scales up the embeddings so their magnitude is comparable to the positional encodings that get added next (positional encodings use sine/cosine values which are bounded between -1 and 1, while raw embeddings can be much smaller — this scaling balances the two).


In [34]:
## input embedding class

# __init__
#input : d_model : dimension of the model ,vocab_size : size of the vocabulary

#forward
#input : x : input tokens of shape (batch_size, seq_len)
#output : embedding of the input tokens
class InputEmbeddings(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * (self.d_model ** 0.5)


#### 3. Positional Encoding

**What this does:** Injects information about the *position* of each token in the sequence into its embedding.

**Why it's needed:** Unlike RNNs, the Transformer has no built-in sense of order — attention treats a sequence like a "bag" of tokens that all look at each other simultaneously. Without extra information, `"dog bites man"` and `"man bites dog"` would look identical to the model. The paper's solution is to add a fixed pattern of sine and cosine waves (of different frequencies) to each token's embedding, so that each position gets a unique "signature" that the model can learn to interpret.

**Key pieces:**
- `__init__(self, d_model, seq_len, dropout)`:
  - `pe = torch.zeros(seq_len, d_model)` — starts with an empty matrix, one row per position, one column per embedding dimension.
  - `position = torch.arange(0, seq_len, ...).unsqueeze(1)` — a column vector `[0, 1, 2, ..., seq_len-1]` representing every position index.
  - `div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))` — this computes the set of frequencies used for the sine/cosine waves, following the formula from the paper: `1 / 10000^(2i/d_model)`. Using `exp(log(...))` is just a numerically stable way to compute a power.
  - `pe[:, 0::2] = sin(position * div_term)` and `pe[:, 1::2] = cos(position * div_term)` — fill even-indexed columns with sine and odd-indexed columns with cosine of `position × frequency`. Every position ends up with a unique combination of wave values across the `d_model` dimensions.
  - `pe.unsqueeze(0)` adds a batch dimension so it can later be broadcast/added to any batch of embeddings, giving shape `(1, seq_len, d_model)`.
  - `self.register_buffer('pe', pe)` — stores `pe` as part of the model's state (so it moves with the model to GPU/CPU and gets saved/loaded), but marks it as **not a trainable parameter** — it's a fixed, precomputed table, not something learned by gradient descent.
- `forward(self, x)`:
  - Simply adds the precomputed positional encoding (sliced to match the current sequence length) to the input embeddings: `x + self.pe[:, :x.size(1), :]`.
  - Applies dropout afterward, as specified in the paper, for regularization.


In [35]:
## creating positional encoding class

# __init__
#input: d_model: dimension of the model, seq_len: maximum sequence length, dropout: dropout rate
# output: positional encoding matrix of shape (seq_len, d_model)

# forward
# input: x: input tensor of shape (batch_size, seq_len, d_model) : (embedding of the input sequence)
# output: tensor of shape (batch_size, seq_len, d_model) with positional encoding added

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        # creating positional encoding matrix (seq_len, d_model)
        pe = torch.zeros(seq_len, d_model)

        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)  ## => [0, 1, 2, ..., seq_len-1] shape (seq_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))  ## shape (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)  ## even indices
        pe[:, 1::2] = torch.cos(position * div_term)  ## odd indices

        pe = pe.unsqueeze(0)  # Add a batch dimension : for shape (1, seq_len, d_model)
        self.register_buffer('pe', pe) # registering the positional encoding matrix as a buffer so that it is not considered a model parameter and will not be updated during training

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :] #positional encoding is not updated during training as this is registered as a buffer and not a parameter of the model
        return self.dropout(x)

#### 4. Layer Normalization

**What this does:** Normalizes the values inside each token's vector so they have (roughly) mean 0 and standard deviation 1, then rescales them with two small learnable parameters.

**Why it's needed:** As data flows through many stacked layers, the scale of the numbers can grow or shrink unpredictably, which makes training unstable. Layer normalization fixes this by normalizing *across the features of a single token* (as opposed to *Batch Normalization*, which normalizes across the batch dimension). This makes it well-suited to sequence models where sequence lengths and batch composition can vary.

**Key pieces:**
- `__init__(self, d_model, eps)`:
  - `self.gamma = nn.Parameter(torch.ones(d_model))` and `self.beta = nn.Parameter(torch.zeros(d_model))` — two learnable vectors (initialized to 1s and 0s) that let the model rescale and shift the normalized output if that turns out to work better than a strict 0-mean/1-std distribution.
  - `eps` — a tiny constant added to avoid ever dividing by zero.
- `forward(self, x)`:
  - `mean = x.mean(dim=-1, keepdim=True)` and `std = x.std(dim=-1, keepdim=True)` — compute the mean and standard deviation *per token*, across its `d_model` features (the last dimension), keeping the dimension so the result can broadcast back against `x`.
  - `x_hat = (x - mean) / (std + eps)` — the actual normalization step.
  - `return self.gamma * x_hat + self.beta` — scale and shift the normalized values using the learnable parameters.


In [36]:
## layer normalization class

#__init__
#input: d_model: dimension of the model, eps: small value to avoid division by zero

#forward
#input: x: input tensor of shape (batch_size, seq_len, d_model)
#output: tensor of shape (batch_size, seq_len, d_model) with layer normalization

class LayerNormalization(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model)) # making it learnable parameter
        self.beta = nn.Parameter(torch.zeros(d_model)) # making it learnable parameter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        # we are calculating the std directly , not the var => this is the small from the actual paper
        
        x_hat = (x - mean) / (std + self.eps) # normalizing the input tensor
        return self.gamma * x_hat + self.beta # scaling and shifting the normalized tensor

#### 5. Position-wise Feed-Forward Network

**What this does:** Applies a small 2-layer neural network independently to every token's vector.

**Why it's needed:** Attention lets tokens exchange information with each other, but it's a fairly simple, linear-ish operation (weighted averaging of value vectors). The feed-forward network gives the model extra capacity to transform each token's representation non-linearly, adding expressive power to the block. It's called "position-wise" because the *exact same* two linear layers are applied to every position in the sequence independently (no mixing across positions happens here — that's attention's job).

**The architecture (as commented in the cell):**
```
x (batch_size, seq_len, d_model)
        ↓
Linear(d_model → d_ff)      # expands to a larger hidden size, e.g. 2048
        ↓
ReLU                        # non-linearity
        ↓
Dropout
        ↓
Linear(d_ff → d_model)      # projects back down to d_model
        ↓
output (batch_size, seq_len, d_model)
```

**Key pieces:**
- `nn.Linear(d_model, d_ff)` then `nn.Linear(d_ff, d_model)` — the two linear (fully-connected) layers; `d_ff` is typically much larger than `d_model` (2048 vs 512 in the paper), giving the network room to learn richer transformations before compressing back down.
- `torch.relu(x)` — the ReLU activation function, `max(0, x)`, applied between the two linear layers as specified in the paper.
- `self.dropout(x)` — randomly zeroes some values during training to reduce overfitting.


In [37]:
## Feed Forward Network class

#architecture : x(batch_size,seq_len,d_model)
#                       ↓
#              Linear(d_model,d_ff) (d_ff=2048 neurons)
#                      ↓
#                    ReLU
#                      ↓
#                  Dropout
#                     ↓
#              Linear(d_ff,d_model) (d_model=512 neurons) 
#                     ↓
#           output(batch_size,seq_len,d_model)  
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

#### 6. Multi-Head Attention

This is the heart of the Transformer. It's worth taking slowly.

**What "attention" means, intuitively:** For every token, attention asks *"which other tokens in this sequence are relevant to me, and how much should I borrow from each of them?"* It does this using three vectors derived from every token:
- **Query (Q)** — "what am I looking for?"
- **Key (K)** — "what do I contain / offer?"
- **Value (V)** — "what information do I actually share if someone attends to me?"

For a given token's Query, the model compares it against every token's Key (via a dot product) to get a similarity **score**. These scores are turned into weights that sum to 1 (via softmax), and the final output is a weighted sum of all the Value vectors — tokens with a higher score contribute more.

**Why "multi-head"?** Instead of doing this once with the full `d_model`-sized vectors, the model splits Q, K, V into `num_heads` smaller chunks (each of size `head_dim = d_model / num_heads`) and runs attention independently, in parallel, on each chunk ("head"). Each head can then specialize in a different kind of relationship (e.g. one head might learn syntax, another might learn coreference). The results from all heads are concatenated back together at the end.

**Key pieces:**
- `__init__(self, d_model, num_heads, dropout)`:
  - Asserts `d_model % num_heads == 0` — `d_model` must split evenly across heads.
  - `self.head_dim = d_model // num_heads` — the size of each head's slice.
  - `w_q`, `w_k`, `w_v` — three separate `nn.Linear(d_model, d_model)` layers that project the raw input into Query, Key, and Value spaces (these are learned — the model figures out during training what makes a good "query" or "key").
  - `w_o` — a final `nn.Linear(d_model, d_model)` that mixes the concatenated multi-head output back together before it leaves this module.
- `attention(query, key, value, mask, dropout)` (a `@staticmethod`, i.e. it doesn't need `self` — it's a pure function of its inputs), implementing **Scaled Dot-Product Attention**:
  - `scores = matmul(query, key.transpose(-2, -1)) / sqrt(head_dim)` — the dot product between every query and every key, scaled down by `sqrt(head_dim)`. This scaling matters because for large `head_dim`, dot products can grow large in magnitude, pushing softmax into regions with tiny gradients — dividing by `sqrt(head_dim)` keeps the scores in a well-behaved range.
  - `if mask is not None: scores = scores.masked_fill(mask == 0, -inf)` — wherever the mask says "0" (i.e. "not allowed to look here"), the score is set to negative infinity so that after softmax its weight becomes 0. This is how padding tokens or "future" tokens get hidden (more on this in the Decoder section).
  - `attention_weights = softmax(scores, dim=-1)` — converts scores into probabilities that sum to 1 across the "key" dimension.
  - `output = matmul(attention_weights, value)` — the actual weighted sum of Value vectors.
  - Returns both the `output` and the `attention_weights` (the latter is useful for later inspection/visualization of what the model is attending to, even though it isn't used further in this notebook).
- `forward(self, query, key, value, mask)`:
  - Projects the raw `query`, `key`, `value` inputs through `w_q`, `w_k`, `w_v`.
  - Reshapes each from `(batch_size, seq_len, d_model)` into `(batch_size, num_heads, seq_len, head_dim)` using `.view(...).transpose(1, 2)` — this is the "split into heads" step. The `.transpose(1, 2)` swaps the sequence and head dimensions so that, per head, we have a clean `(seq_len, head_dim)` slice to run attention on.
  - Calls the static `attention` method to get the per-head outputs.
  - Reshapes back with `.transpose(1, 2).contiguous().view(...)` — the inverse operation, concatenating all heads back into a single `(batch_size, seq_len, d_model)` tensor. `.contiguous()` is needed because `.transpose` can leave the tensor's memory layout non-contiguous, which `.view` requires.
  - Passes the result through `w_o`, the final output projection.


In [38]:
# multi head attention class

# __init__
# input: d_model: dimension of the model, num_heads: number of attention heads, dropout: dropout rate
# creating the multi head attention layer with linear projections for query, key, value and output

# attention
# input: query, key, value: input tensors of shape (batch_size, num_heads, seq_len, head_dim)
#        mask: optional mask tensor of shape (batch_size, seq_len, seq_len)
#        dropout: dropout layer for attention weights
# output : the attention score of shape (batch_size, num_heads, seq_len, head_dim) and 
#          the attention weights ( softmax ( if mask then mask(sclaled dot product of query and key) 
#                                           else softmax(scaled dot product of query and key)  )


# forward
# input: query, key, value: input tensors of shape (batch_size, seq_len, d_model)
#        mask: optional mask tensor of shape (batch_size, seq_len, seq_len)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.head_dim = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor, dropout: nn.Dropout) -> torch.Tensor:
        # query, key, value: (batch_size, num_heads, seq_len, head_dim)
        head_dim = query.size(-1)

        # Scaled dot-product attention
        #  query , key : (batch_size, num_heads, seq_len_q, head_dim) , (batch_size, num_heads, seq_len_k, head_dim)
        #  key.transpose(-2, -1) : (batch_size, num_heads, head_dim, seq_len_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(head_dim)  # scores : (batch_size, num_heads, seq_len_q, seq_len_k)

        # for masked multihead attention
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attention_weights = torch.softmax(scores, dim=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)
        if dropout is not None:
            attention_weights = dropout(attention_weights)

        output = torch.matmul(attention_weights, value)  # (batch_size, num_heads, seq_len_q, head_dim)
        return output , attention_weights

    def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # query, key, value: (batch_size, seq_len, d_model)
        batch_size = query.size(0)
        seq_len_q = query.size(1)
        seq_len_k = key.size(1)
        seq_len_v = value.size(1)

        # Linear projections
        Q = self.w_q(query)  # (batch_size, seq_len, d_model)
        K = self.w_k(key)      # (batch_size, seq_len, d_model)
        V = self.w_v(value)  # (batch_size, seq_len, d_model)

        # Split into multiple heads
        # from each head , we want to get the sequences => so, we have transposed the views
        # after transposing , now we can have like this : from every head => we have all the words in the sequence of the head_dim dimensions 
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len_q, head_dim)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len_k, head_dim)
        V = V.view(batch_size, seq_len_v, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len_v, head_dim)

        output, attention_weights = self.attention(Q, K, V, mask, self.dropout)
        
        # output: (batch_size, num_heads, seq_len, head_dim)
        # for geting the concatenated output of all the heads, first we need to transpose the output to (batch_size, seq_len, num_heads, head_dim)
        # before passing through the output linear layer, we need to reshape it back to (batch_size, seq_len, d_model)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.d_model)  # (batch_size, seq_len_q, d_model)
        output = self.w_o(output)  # (batch_size, seq_len_q, d_model)

        return output

#### 7. Residual Connection (Add & Norm)

**What this does:** Wraps a sublayer (like multi-head attention or the feed-forward network) with a **skip connection** and a **layer normalization**, following the pattern used throughout the paper: `input → sublayer → dropout → add input back → layer norm → output`.

**Why it's needed:** As networks get deeper (this model stacks many of these), gradients can struggle to flow all the way back to earlier layers during training ("vanishing gradients"), and it becomes harder for a layer to simply "do nothing" if that's the best option. A **residual/skip connection** — literally adding the original input `x` back onto the sublayer's output — gives gradients a direct path backward and makes it trivial for a layer to learn something close to the identity function if needed. This is the same idea popularized by ResNets in computer vision.

**Key pieces:**
- `self.layer_norm = LayerNormalization(d_model)` and `self.dropout = nn.Dropout(dropout)` — the normalization and dropout components used in the wrapper.
- `forward(self, x, sublayer)`:
  - `sublayer` is passed in as a **function** (often a `lambda`), not a fixed layer — this makes `ResidualConnection` generic enough to wrap *either* the attention block *or* the feed-forward block, since both just need to take `x` and return a tensor of the same shape.
  - `sublayer_output = sublayer(x)` — runs whatever sublayer was passed in.
  - `return self.layer_norm(x + self.dropout(sublayer_output))` — adds the original input back (`x +`) to the (dropout-ed) sublayer output, then normalizes the sum.


In [39]:
# residual connection class

# the flow will be like this : input -> sublayer -> dropout -> add input -> layer normalization -> output ( as in the original paper)
class ResidualConnection(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.layer_norm = LayerNormalization(d_model)
        self.dropout = nn.Dropout(dropout)

    # here sublayer is a function that takes input x and returns output of the sublayer 
    # the function can be => multihead attention or feed forward network
    def forward(self, x, sublayer): 
        sublayer_output = sublayer(x)
        return self.layer_norm(x + self.dropout(sublayer_output))

#### 8. Encoder Block

**What this does:** Combines the pieces above into one full encoder layer, matching this structure from the paper:

```
input (embeddings + positional encoding)
            ↓
Residual Connection 1  (sublayer = Multi-Head Self-Attention)
            ↓
Residual Connection 2  (sublayer = Feed-Forward Network)
            ↓
output → next encoder block (or the decoder, if this is the last one)
```

This is "self-attention" because the Query, Key, and Value all come from the *same* sequence (`x`) — every source token attends to every other source token.

**Key pieces:**
- `__init__` creates one `MultiHeadAttention`, one `FeedForwardNetwork`, and **two separate** `ResidualConnection` wrappers (one per sublayer — each needs its own layer norm/dropout, since they wrap different sublayers).
- `forward(self, x, mask)`:
  - `self.residual_connection1(x, lambda x: self.multi_head_attention(x, x, x, mask))` — wraps self-attention in the first residual connection. The `lambda` is needed because `ResidualConnection.forward` expects a one-argument function, but `multi_head_attention` needs query/key/value/mask — the lambda "bakes in" `x` as query, key, *and* value, plus the mask, leaving just `x` as the free argument that `ResidualConnection` will call.
  - `self.residual_connection2(x, self.feed_forward)` — wraps the feed-forward network in the second residual connection. No lambda is needed here since `feed_forward` already takes a single tensor argument.


In [40]:
# encoder block class
# for each encoder block:
#                       input ( positional encoding + input embedding )
#                                   ↓
#                      residual connection_1 (sublayer function =  multi head attention )
#                                   ↓
#                      residual connection_2 (sublayer function =  feed forward network )
#                                   ↓
#                      output ( to the next encoder block or decoder )

class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)
        self.residual_connection1 = ResidualConnection(d_model, dropout)
        self.residual_connection2 = ResidualConnection(d_model, dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        # x: (batch_size, seq_len, d_model)
        # mask: (batch_size, seq_len, seq_len)
        x = self.residual_connection1(x, lambda x: self.multi_head_attention(x, x, x, mask)) # lambda function is used to pass the sublayer function to the residual connection
        x = self.residual_connection2(x, self.feed_forward)
        return x

#### 9. Encoder (stack of Encoder Blocks)

**What this does:** Stacks `num_layers` copies of `EncoderBlock` on top of each other (6 in the original paper) — the output of one block becomes the input to the next — and applies one final layer normalization at the very end.

**Key pieces:**
- `self.layers = nn.ModuleList([EncoderBlock(...) for _ in range(num_layers)])` — `nn.ModuleList` is used (instead of a plain Python list) so that PyTorch correctly registers all the sub-modules' parameters (this matters for things like `.to(device)`, saving/loading, and optimizer parameter discovery).
- `forward(self, x, mask)`:
  - Loops through every block, feeding each one's output into the next: `x = layer(x, mask)`.
  - After the loop, applies one more `LayerNormalization` to the final output — a common convention (sometimes called a "final norm") that stabilizes the representation handed off to the decoder.


In [41]:
# encoder layer class

# input : num_layers : number of encoder blocks , d_model : dimension of the model , num_heads : number of attention heads , d_ff : dimension of the feed forward network , dropout : dropout rate
# output : tensor of shape (batch_size, seq_len, d_model) after passing through all the encoder blocks
class Encoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask) # each encoder block forwards the output to the next encoder block
        x = self.layer_norm(x)
        return x

#### 10. Decoder Block

**What this does:** The decoder's building block, structurally similar to the encoder block but with **three** sublayers instead of two:

```
input (target embeddings + positional encoding)
            ↓
Residual Connection 1  (sublayer = Masked Multi-Head Self-Attention)
            ↓
Residual Connection 2  (sublayer = Cross Multi-Head Attention)
            ↓
Residual Connection 3  (sublayer = Feed-Forward Network)
            ↓
output → next decoder block (or the final output projection)
```

**Why two different attentions?**
- **Masked self-attention** — the decoder attends to the target sequence *generated so far*. It's "masked" because, during training, the model is shown the whole target sentence at once for efficiency, but it must not be allowed to "cheat" by looking at future tokens it hasn't generated yet. The `tgt_mask` blocks out (sets to `-inf` before softmax) any position that comes after the current one, as well as padding.
- **Cross-attention** — here, the Query comes from the decoder (the target sequence so far), while the Key and Value come from the **encoder's output** (the source sequence). This is the mechanism that actually lets the decoder "look back" at the input sentence while generating the output — e.g. attending to the right English words while producing each French word.

**Key pieces:**
- `self.masked_multi_head_attention` and `self.cross_multi_head_attention` — two separate `MultiHeadAttention` instances (different learned weights, since they play different roles), plus `self.feed_forward` and three `ResidualConnection` wrappers (one per sublayer, same reasoning as the encoder block).
- `forward(self, x, encoder_output, src_mask, tgt_mask)`:
  - `x` — the target sequence representation so far, shape `(batch_size, seq_len_tgt, d_model)`.
  - `encoder_output` — the encoder's final output for the source sequence, shape `(batch_size, seq_len_src, d_model)`.
  - `residual_connection1(x, lambda x: self.masked_multi_head_attention(x, x, x, tgt_mask))` — self-attention over the target, using `tgt_mask` to hide future tokens/padding.
  - `residual_connection2(x, lambda x: self.cross_multi_head_attention(x, encoder_output, encoder_output, src_mask))` — cross-attention: notice `x` is the Query, while `encoder_output` is used as *both* Key and Value; `src_mask` hides source-side padding.
  - `residual_connection3(x, self.feed_forward)` — the same position-wise feed-forward network as in the encoder.


In [42]:
# decoder block class
# for each decoder block:
#                       input ( positional encoding + input embedding )
#                                   ↓
#                      residual connection_1 (sublayer function =  masked multi head attention )
#                                   ↓
#                      residual connection_2 (sublayer function = cross multi head attention )
#                                   ↓
#                      residual connection_3 (sublayer function =  feed forward network )
#                                   ↓
#                        output ( to the next decoder block or final linear layer )

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.masked_multi_head_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_multi_head_attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)
        self.residual_connection1 = ResidualConnection(d_model, dropout)
        self.residual_connection2 = ResidualConnection(d_model, dropout)
        self.residual_connection3 = ResidualConnection(d_model, dropout)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: torch.Tensor = None, tgt_mask: torch.Tensor = None) -> torch.Tensor:
        # x: (batch_size, seq_len_tgt, d_model)
        # encoder_output: (batch_size, seq_len_src, d_model)
        # src_mask: (batch_size, seq_len_src, seq_len_src) => Hide padding/invalid source tokens 
        # tgt_mask: (batch_size, seq_len_tgt, seq_len_tgt) => hide future tokens and padding/invalid target tokens
        x = self.residual_connection1(x, lambda x: self.masked_multi_head_attention(x, x, x, tgt_mask)) # lambda function is used to pass the sublayer function to the residual connection
        x = self.residual_connection2(x, lambda x: self.cross_multi_head_attention(x, encoder_output, encoder_output, src_mask)) # lambda function is used to pass the sublayer function to the residual connection
        x = self.residual_connection3(x, self.feed_forward)
        return x

#### 11. Decoder (stack of Decoder Blocks)

**What this does:** Exactly analogous to the `Encoder` class — stacks `num_layers` `DecoderBlock`s, passing each one's output (along with the fixed `encoder_output`, `src_mask`, and `tgt_mask`) into the next, then applies a final layer normalization.

**Key pieces:**
- Same `nn.ModuleList` pattern as the `Encoder`.
- `forward(self, x, encoder_output, src_mask, tgt_mask)` loops through every decoder block, threading `encoder_output`, `src_mask`, and `tgt_mask` through unchanged at every layer (only `x`, the decoder's own running representation, gets updated block to block).


In [43]:
# decoder layer class

class Decoder(nn.Module):
    def __init__(self, num_layers: int, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: torch.Tensor = None, tgt_mask: torch.Tensor = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask) # each decoder block forwards the output to the next decoder block
        x = self.layer_norm(x)
        return x

#### 12. Output Projection (Linear + Softmax)

**What this does:** Converts the decoder's final `d_model`-sized vectors into a probability distribution over the entire output vocabulary — i.e. for each position, "how likely is each possible word to come next?"

**Key pieces:**
- `self.linear = nn.Linear(d_model, vocab_size)` — a single linear layer that projects from `d_model` up to `vocab_size` — one raw score ("logit") per possible token.
- `forward(self, x)`:
  - `torch.log_softmax(self.linear(x), dim=-1)` — applies softmax (turning logits into probabilities that sum to 1 across the vocabulary) in log-space. **Log-softmax** is used instead of plain softmax because it pairs naturally and numerically stably with `nn.NLLLoss` (negative log-likelihood loss), a common choice for training this kind of token-prediction model.
  - Output shape: `(batch_size, seq_len, vocab_size)` — a distribution over the vocabulary at every position in the sequence.


In [44]:
# projecting the output of the decoder to the vocabulary size for generating the final output probabilities

#input : d_model : dimension of the model , vocab_size : size of the vocabulary
#output : tensor of shape (batch_size, seq_len, vocab_size) after passing through the linear layer and log softmax activation function
# forward : input : x : input : tensor of shape (batch_size, seq_len, d_model) ,
#                       output : tensor of shape (batch_size, seq_len, vocab_size) 
class ProjectionToVocab(nn.Module):
    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()
        self.linear = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.log_softmax(self.linear(x), dim=-1)  # output shape: (batch_size, seq_len, vocab_size)


#### 13. The Full Transformer

**What this does:** Wires every piece built so far together into the complete encoder–decoder model from the paper.

**Key pieces:**
- `__init__`: builds one instance of every component —
  - `self.input_embedding` / `self.output_embedding` — **separate** embedding tables for the source and target vocabularies (they can even be different languages with different vocab sizes).
  - `self.positional_encoding` — a single shared `PositionalEncoding` module, reused for both source and target sequences (since the sinusoidal formula only depends on position and `d_model`, not on which sequence it's applied to).
  - `self.encoder` / `self.decoder` — the full stacks built earlier.
  - `self.projection_to_vocab` — the final linear+log-softmax layer.
  - `self._init_weights()` is called at the end of `__init__`.
- `_init_weights(self)`: loops over every parameter in the model and applies **Xavier (Glorot) uniform initialization** to any parameter with more than 1 dimension (i.e. weight matrices, not the 1-D bias/gamma/beta vectors). Good weight initialization helps gradients flow well from the very first training step, rather than the model starting from arbitrary/poorly-scaled random values.
- `encode(self, src, src_mask)`: runs the full **encoder side** — embed the source tokens, add positional encoding, then pass through the encoder stack. Returns the encoder's output, which will later be reused (unchanged) for every decoding step.
- `decode(self, tgt, src_encoded, src_mask, tgt_mask)`: runs the full **decoder side** — embed the target tokens, add positional encoding, then pass through the decoder stack, using `src_encoded` (the encoder's output) for cross-attention.
- `forward(self, src, tgt, src_mask, tgt_mask)`: the standard training-time forward pass — `encode` the source once, `decode` the target using that encoding, then project to vocabulary probabilities with `self.projection_to_vocab`. (Note `encode` and `decode` are also exposed as separate methods, which is useful at inference time — you compute `encode` once for the source sentence, then call `decode` repeatedly, one new token at a time, without redoing the encoder pass.)


In [45]:
class Transformer(nn.Module):

    def __init__(
        self,
        num_encoder_layers: int,
        num_decoder_layers: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        input_vocab_size: int,
        output_vocab_size: int,
        max_seq_length: int,
        dropout: float = 0.1
    ):
        super().__init__()

        # Separate embedding layers
        self.input_embedding = InputEmbeddings(
            d_model,
            input_vocab_size
        )

        self.output_embedding = InputEmbeddings(
            d_model,
            output_vocab_size
        )

        self.positional_encoding = PositionalEncoding(
            d_model,
            max_seq_length
        )

        self.encoder = Encoder(
            num_encoder_layers,
            d_model,
            num_heads,
            d_ff,
            dropout
        )

        self.decoder = Decoder(
            num_decoder_layers,
            d_model,
            num_heads,
            d_ff,
            dropout
        )

        self.projection_to_vocab = ProjectionToVocab(
            d_model,
            output_vocab_size
        )

        self._init_weights()

    # check for the initialization of the weights of the model parameters using Xavier uniform initialization for parameters with more than one dimension.
    def _init_weights(self):

        for parameter in self.parameters():

            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    def encode(
        self,
        src: torch.Tensor,
        src_mask: torch.Tensor = None
    ) -> torch.Tensor:

        src_embedded = self.input_embedding(src)
        # (batch_size, seq_len_src, d_model)

        src_positional_encoded = self.positional_encoding(src_embedded)
        # (batch_size, seq_len_src, d_model)

        src_encoded = self.encoder(
            src_positional_encoded,
            src_mask
        )
        # (batch_size, seq_len_src, d_model)

        return src_encoded

    def decode(
        self,
        tgt: torch.Tensor,
        src_encoded: torch.Tensor,
        src_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None
    ) -> torch.Tensor:

        tgt_embedded = self.output_embedding(tgt)
        # (batch_size, seq_len_tgt, d_model)

        tgt_positional_encoded = self.positional_encoding(tgt_embedded)
        # (batch_size, seq_len_tgt, d_model)

        tgt_decoded = self.decoder(
            tgt_positional_encoded,
            src_encoded,
            src_mask,
            tgt_mask
        )
        # (batch_size, seq_len_tgt, d_model)

        return tgt_decoded

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None
    ) -> torch.Tensor:

        # src: (batch_size, seq_len_src)
        # tgt: (batch_size, seq_len_tgt)

        # src_mask:
        # (batch_size, seq_len_src, seq_len_src)

        # tgt_mask:
        # (batch_size, seq_len_tgt, seq_len_tgt)

        src_encoded = self.encode(
            src,
            src_mask
        )

        tgt_decoded = self.decode(
            tgt,
            src_encoded,
            src_mask,
            tgt_mask
        )

        output = self.projection_to_vocab(tgt_decoded)
        # (batch_size, seq_len_tgt, output_vocab_size)

        return output

#### 14. `BuildTransformer` — a Convenience Builder

**What this does:** A small helper/factory class that just takes the same hyperparameters as `Transformer.__init__`, constructs a `Transformer` instance internally, and exposes it via `get_model()`.

This is a common pattern for keeping model-construction code tidy — e.g. it makes it easy to plug hyperparameter configs (from a dictionary, config file, or CLI arguments) into one place and get back a ready-to-use model with a single call, without the calling code needing to know about `Transformer`'s constructor directly.


In [46]:
class BuildTransformer:

    def __init__(
        self,
        num_encoder_layers: int,
        num_decoder_layers: int,
        d_model: int,
        num_heads: int,
        d_ff: int,
        input_vocab_size: int,
        output_vocab_size: int,
        max_seq_length: int,
        dropout: float = 0.1
    ):

        self.transformer = Transformer(
            num_encoder_layers,
            num_decoder_layers,
            d_model,
            num_heads,
            d_ff,
            input_vocab_size,
            output_vocab_size,
            max_seq_length,
            dropout
        )

    def get_model(self):
        return self.transformer

---
### Summary

Putting it all together, this is the full data flow of the model implemented above:

1. **Source tokens** → `InputEmbeddings` → `+ PositionalEncoding` → **Encoder** (stack of `EncoderBlock`s, each doing self-attention + feed-forward, wrapped in residual connections) → `src_encoded`.
2. **Target tokens** (so far) → `InputEmbeddings` → `+ PositionalEncoding` → **Decoder** (stack of `DecoderBlock`s, each doing masked self-attention + cross-attention with `src_encoded` + feed-forward) → `tgt_decoded`.
3. `tgt_decoded` → `ProjectionToVocab` → probability distribution over the next token.


